In [24]:
# ============================================================
# Cell 1 — Imports + CONFIG 
# ============================================================
import os
import re
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import transform_bounds
from rasterio.mask import mask
from shapely.geometry import box
from tqdm import tqdm

# --- PATHS ---
FLOODS_ROOT = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered"  # folder containing year subfolders
COMMUNES_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025.gpkg"
COMMUNES_LAYER = "commune"  # IMPORTANT: use polygon layer

OUT_WIDE = os.path.join(FLOODS_ROOT, "events_communes_wide.csv")
OUT_LONG = os.path.join(FLOODS_ROOT, "events_communes_long.csv")

# --- SEMANTICS ---
DEPTH_THRESHOLD_CM = 0.0            # commune affected if any pixel depth > this
IGNORE_VALUES = {0.0, 9999.0}       # 0=no flood, 9999=permanent/seasonal water

# Optional: strictness of "touch" on borders
ALL_TOUCHED = True

# Quick sanity check: list a few TIFs
tifs_preview = sorted(glob.glob(os.path.join(FLOODS_ROOT, "**", "*.tif"), recursive=True))[:5]
print("Preview TIFs:", *tifs_preview, sep="\n- ")
print("\nWill write:\n-", OUT_WIDE, "\n-", OUT_LONG)


Will scan flood roots:
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2022_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2023_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2015_filtered

Will write combined outputs to:
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs\events_communes_wide_all_years.csv
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs\events_communes_long_all_years.csv


In [25]:
# ============================================================
# Cell 2 — Helpers (filename parsing + commune columns + test)
# ============================================================

# Filename pattern:
# WD_MERGE_[start]---[end]_duration_[d]_days_cluster_[id]_...
PAT = re.compile(
    r"WD_MERGE_(\d{4}-\d{2}-\d{2})---(\d{4}-\d{2}-\d{2})_duration_(\d+)_days_cluster_([^_]+)_.*\.tif$",
    re.IGNORECASE
)


#m.groups() simply returns a tuple containing the text matched by each group using the ()

def parse_event_from_name(path: str):
    name = os.path.basename(path)
    m = PAT.match(name)
    if not m:
        return None
    start_date, end_date, duration_days, cluster_id = m.groups()
    return {
        "event_file": name,
        "event_path": path,
        "start_date": start_date,
        "end_date": end_date,
        "duration_days": int(duration_days),
        "cluster_id": str(cluster_id),
    }

def pick_columns(communes: gpd.GeoDataFrame):
    """
    Your AdminExpress simplified gpkg has:
      - insee_com  (commune INSEE code)   use this
      - nom        (commune name)         use this
    """
    preferred_code = ["insee_com", "INSEE_COM", "codgeo", "CODGEO"]
    preferred_name = ["nom", "NOM", "nom_com", "NOM_COM", "libelle", "LIBELLE"]

    code_col = next((c for c in preferred_code if c in communes.columns), None)
    name_col = next((c for c in preferred_name if c in communes.columns), None)

    if code_col is None or name_col is None:
        raise ValueError(
            f"Couldn't find commune code/name columns. Available: {list(communes.columns)}"
        )
    return code_col, name_col

#Cut out the part of the flood raster that lies inside this commune
def commune_is_affected(src, geom, threshold: float):
    # returns masked array when filled=False
    # _ "I receive this value, but I deliberately ignore it"
    data, _ = mask(src, [geom], crop=True, filled=False, all_touched=ALL_TOUCHED, indexes=1)
    arr = data[0].astype(np.float32, copy=False)

    vals = arr.compressed() if np.ma.isMaskedArray(arr) else arr.ravel()
    if vals.size == 0:
        return False, None

    # filter ignore values
    for v in IGNORE_VALUES:
        vals = vals[vals != v]
    if vals.size == 0:
        return False, None

    max_cm = float(np.max(vals))
    return (max_cm > threshold), max_cm


# --- Load communes + build spatial index (this will fail fast if anything is wrong) ---
communes = gpd.read_file(COMMUNES_GPKG, layer=COMMUNES_LAYER)
if communes.crs is None:
    raise ValueError("Communes layer has no CRS.")

code_col, name_col = pick_columns(communes)
print("Using columns:", code_col, name_col)

communes_wgs84 = communes.to_crs("EPSG:4326")
sindex = communes_wgs84.sindex  # requires rtree/pygeos; if it errors, install rtree

print("Communes loaded:", len(communes_wgs84))
print("CRS:", communes_wgs84.crs)
print(communes_wgs84[[code_col, name_col]].head())


Using columns: insee_com nom
Communes loaded: 34877
CRS: EPSG:4326
  insee_com         nom
0     70177       Corre
1     89050     Bonnard
2     60414  Montataire
3     80756    Thièvres
4     14478       Orbec


In [27]:
# ============================================================
# Cell 3 — FULL PIPELINE (multi-root, all years combined)
# ============================================================

def infer_year_from_path(fp: str):
    # Tries to infer year from folder names like '2023_filtered' or from the filename itself (start_date)
    parts = fp.replace("/", os.sep).split(os.sep)
    for p in parts[::-1]:
        m = re.search(r"(19\d{2}|20\d{2})", p)
        if m:
            return int(m.group(1))
    # fallback: parse start_date
    meta = parse_event_from_name(fp)
    if meta:
        return int(meta["start_date"][:4])
    return None

# Collect all TIFs across roots
all_tifs = []
for root in FLOODS_ROOTS:
    tifs = sorted(glob.glob(os.path.join(root, "**", "*.tif"), recursive=True))
    all_tifs.extend(tifs)

print("Total TIFs found (all roots):", len(all_tifs))

long_rows = []
wide_rows = []
errors = []
event_diag = {
    "events_total": 0,
    "events_parsed": 0,
    "events_with_candidates": 0,
    "events_with_affected": 0,
}

for fp in tqdm(all_tifs, desc="Processing events"):
    event_diag["events_total"] += 1
    meta = parse_event_from_name(fp)
    if meta is None:
        continue
    event_diag["events_parsed"] += 1

    # Add a year column (useful for grouping later)
    meta = {**meta, "year": infer_year_from_path(fp)}

    try:
        with rasterio.open(fp) as src:
            if src.crs is None:
                errors.append((fp, "ValueError('Raster has no CRS')"))
                continue

            # Raster bounds in WGS84 for fast candidate filtering
            b = transform_bounds(src.crs, "EPSG:4326", *src.bounds, densify_pts=21)
            raster_bbox = box(*b)

            cand_idx = list(sindex.intersection(raster_bbox.bounds))
            if not cand_idx:
                wide_rows.append({**meta, "n_communes": 0, "communes": ""})
                continue

            cands = communes_wgs84.iloc[cand_idx]
            cands = cands[cands.intersects(raster_bbox)]
            if cands.empty:
                wide_rows.append({**meta, "n_communes": 0, "communes": ""})
                continue

            event_diag["events_with_candidates"] += 1

            # Reproject candidates to raster CRS for masking
            cands_src = cands.to_crs(src.crs)

            affected = []
            # IMPORTANT: catch per-commune errors so one bad geometry doesn't kill the event
            for _, row in cands_src.iterrows():
                try:
                    ok, max_val = commune_is_affected(src, row.geometry, DEPTH_THRESHOLD)
                except Exception as e:
                    # log and skip this commune
                    errors.append((fp, f"commune_error: {repr(e)}"))
                    continue

                if ok:
                    insee = row[code_col]
                    nom = row[name_col]
                    affected.append((insee, nom, max_val))
                    long_rows.append({
                        **meta,
                        "insee_com": insee,
                        "commune": nom,
                        "max_depth": max_val,
                    })

            affected_sorted = sorted(affected, key=lambda x: (str(x[1]), str(x[0])))
            communes_str = "; ".join([f"{nom} ({insee})" for insee, nom, _ in affected_sorted])

            wide_rows.append({
                **meta,
                "n_communes": len(affected_sorted),
                "communes": communes_str,
            })

            if affected_sorted:
                event_diag["events_with_affected"] += 1

            if VERBOSE:
                print("\nEVENT:", os.path.basename(fp))
                print(" - candidates:", len(cands_src), "affected:", len(affected_sorted))
                if affected_sorted:
                    print(" - max_depth(max over affected communes):", max(x[2] for x in affected_sorted))

    except Exception as e:
        errors.append((fp, repr(e)))
        continue

# Build DataFrames (robust even if empty)
wide_cols = ["year","event_file","event_path","start_date","end_date","duration_days","cluster_id","n_communes","communes"]
long_cols = ["year","event_file","event_path","start_date","end_date","duration_days","cluster_id","insee_com","commune","max_depth"]

df_wide = pd.DataFrame(wide_rows, columns=wide_cols)
df_long = pd.DataFrame(long_rows, columns=long_cols)

if not df_wide.empty:
    df_wide = df_wide.sort_values(["year","start_date", "end_date", "cluster_id"])
if not df_long.empty:
    df_long = df_long.sort_values(["year","start_date", "end_date", "cluster_id", "commune"])

df_wide.to_csv(OUT_WIDE_ALL, index=False, encoding="utf-8-sig")
df_long.to_csv(OUT_LONG_ALL, index=False, encoding="utf-8-sig")

print("\nSaved combined outputs:")
print(" -", OUT_WIDE_ALL, "rows:", len(df_wide))
print(" -", OUT_LONG_ALL, "rows:", len(df_long))

print("\nDiagnostics:")
for k,v in event_diag.items():
    print(f" - {k}: {v}")
print(" - errors logged:", len(errors))

# Optional: quick preview
display(df_wide.head(10))
display(df_long.head(10))

# Optional: review errors (first 15)
if errors:
    print("\nErrors (first 15):")
    for fp, err in errors[:15]:
        print("-", os.path.basename(fp), "->", err)



Total TIFs found (all roots): 573


Processing events: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 573/573 [40:49<00:00,  4.27s/it]


Saved combined outputs:
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs\events_communes_wide_all_years.csv rows: 573
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs\events_communes_long_all_years.csv rows: 780

Diagnostics:
 - events_total: 573
 - events_parsed: 573
 - events_with_candidates: 141
 - events_with_affected: 39
 - errors logged: 302255


,year,event_file,event_path,start_date,end_date,duration_days,cluster_id,n_communes,communes
404,2014,WD_MERGE_2014-12-29---2015-01-12_duration_14_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2014-12-29,2015-01-12,14,001,0,
405,2014,WD_MERGE_2014-12-29---2015-01-12_duration_14_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2014-12-29,2015-01-12,14,002,0,
406,2014,WD_MERGE_2014-12-29---2015-02-09_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2014-12-29,2015-02-09,35,333,0,
407,2015,WD_MERGE_2015-01-05---2015-01-12_duration_7_da...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-05,2015-01-12,7,009,0,
408,2015,WD_MERGE_2015-01-05---2015-01-12_duration_7_da...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-05,2015-01-12,7,010,0,
409,2015,WD_MERGE_2015-01-05---2015-01-12_duration_7_da...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-05,2015-01-12,7,011,0,
410,2015,WD_MERGE_2015-01-05---2015-01-12_duration_7_da...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-05,2015-01-12,7,012,0,
411,2015,WD_MERGE_2015-01-12---2015-01-26_duration_14_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-12,2015-01-26,14,015,2,Bouquemont (55064); Rouvrois-sur-Meuse (55444)
412,2015,WD_MERGE_2015-01-12---2015-02-23_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-12,2015-02-23,35,016,5,Pouan-les-Vallées (10299); Rances (10315); Sai...
413,2015,WD_MERGE_2015-01-19---2015-02-09_duration_21_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-19,2015-02-09,21,020,16,Brixey-aux-Chanoines (55080); Champdolent (170...


,year,event_file,event_path,start_date,end_date,duration_days,cluster_id,insee_com,commune,max_depth
695,2015,WD_MERGE_2015-01-12---2015-01-26_duration_14_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-12,2015-01-26,14,015,55064,Bouquemont,261.0
694,2015,WD_MERGE_2015-01-12---2015-01-26_duration_14_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-12,2015-01-26,14,015,55444,Rouvrois-sur-Meuse,15.0
697,2015,WD_MERGE_2015-01-12---2015-02-23_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-12,2015-02-23,35,016,10299,Pouan-les-Vallées,14.0
696,2015,WD_MERGE_2015-01-12---2015-02-23_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-12,2015-02-23,35,016,10315,Rances,18.0
698,2015,WD_MERGE_2015-01-12---2015-02-23_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-12,2015-02-23,35,016,51502,Saint-Martin-aux-Champs,10.0
700,2015,WD_MERGE_2015-01-12---2015-02-23_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-12,2015-02-23,35,016,51533,Servon-Melzicourt,39.0
699,2015,WD_MERGE_2015-01-12---2015-02-23_duration_35_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-12,2015-02-23,35,016,51610,Verrières,28.0
714,2015,WD_MERGE_2015-01-19---2015-02-09_duration_21_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-19,2015-02-09,21,020,55080,Brixey-aux-Chanoines,10.0
707,2015,WD_MERGE_2015-01-19---2015-02-09_duration_21_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-19,2015-02-09,21,020,17085,Champdolent,182.0
711,2015,WD_MERGE_2015-01-19---2015-02-09_duration_21_d...,D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flo...,2015-01-19,2015-02-09,21,020,70267,Gevigney-et-Mercey,194.0



Errors (first 15):
- WD_MERGE_2022-01-31---2022-03-07_duration_28_days_cluster_336_A0_002048_A_002556_lat_05674_lon_01858_size_0509.tif -> commune_error: ValueError('Input shapes do not overlap raster.')
- WD_MERGE_2022-01-31---2022-03-07_duration_28_days_cluster_336_A0_002048_A_002556_lat_05674_lon_01858_size_0509.tif -> commune_error: ValueError('Input shapes do not overlap raster.')
- WD_MERGE_2022-01-31---2022-03-07_duration_28_days_cluster_336_A0_002048_A_002556_lat_05674_lon_01858_size_0509.tif -> commune_error: ValueError('Input shapes do not overlap raster.')
- WD_MERGE_2022-01-31---2022-03-07_duration_28_days_cluster_336_A0_002048_A_002556_lat_05674_lon_01858_size_0509.tif -> commune_error: ValueError('Input shapes do not overlap raster.')
- WD_MERGE_2022-01-31---2022-03-07_duration_28_days_cluster_336_A0_002048_A_002556_lat_05674_lon_01858_size_0509.tif -> commune_error: ValueError('Input shapes do not overlap raster.')
- WD_MERGE_2022-01-31---2022-03-07_duration_28_days_clu

In [28]:
from collections import Counter

# errors is a list of tuples like (fp, commune_id/name, repr(e)) in the modified notebook
print("Total errors:", len(errors))

# show top 10 error messages
msg_counts = Counter(err for *_, err in errors)
for msg, n in msg_counts.most_common(10):
    print(n, msg)


Total errors: 302255
302255 commune_error: ValueError('Input shapes do not overlap raster.')


In [ ]:
# ============================================================
# 3_Tabularization — FULL PIPELINE (multi FLOODS_ROOTS)
# Builds:
#   - events_communes_wide_all_years.csv
#   - events_communes_long_all_years.csv
# ============================================================

import os
import re
import glob
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_bounds
from shapely.geometry import box
from tqdm import tqdm

# =========================
# CONFIG
# =========================
FLOODS_ROOTS = [
    r"data/JRC_flood_depth_maps/2022_filtered",
    r"data/JRC_flood_depth_maps/2023_filtered",
    r"data/JRC_flood_depth_maps/2024_filtered",
    # add more...
]

COMMUNES_PATH = r"data/communes/communes.gpkg"   # change if needed
COMMUNES_LAYER = None                            # e.g. "communes" if needed, else None
COMMUNE_NAME_COL = "NOM"                         # change to your column (e.g. "NAME")
COMMUNE_CODE_COL = "INSEE_COM"                   # change to your column (optional)

OUT_DIR = r"data/JRC_flood_depth_maps/_outputs"
OUT_WIDE = os.path.join(OUT_DIR, "events_communes_wide_all_years.csv")
OUT_LONG = os.path.join(OUT_DIR, "events_communes_long_all_years.csv")

# Flood depth threshold (centimeters)
THRESHOLD_CM = 5.0

# Mask behavior
ALL_TOUCHED = True

# If your rasters have special nodata/sentinel values, list them here
IGNORE_VALUES = set()  # e.g. {0, -9999}

# ============================================================
# Helpers
# ============================================================

def infer_year_from_path(fp: str) -> int | None:
    # Try folder year first (e.g. .../2023_filtered/...)
    m = re.search(r"(19|20)\d{2}", fp)
    return int(m.group(0)) if m else None

def parse_event_from_name(fp: str) -> dict | None:
    """
    Expects filenames like:
      WD_MERGE_2024-12-16---2024-12-23_duration_7_days_cluster_12.tif
    Adjust regex if your naming differs.
    """
    base = os.path.basename(fp)

    # flexible cluster id
    m = re.search(
        r"(?P<start>\d{4}-\d{2}-\d{2})---(?P<end>\d{4}-\d{2}-\d{2}).*?"
        r"(?:cluster[_-](?P<cluster>\d+))?",
        base
    )
    if not m:
        return None

    start_date = m.group("start")
    end_date = m.group("end")
    cluster_id = m.group("cluster")
    cluster_id = int(cluster_id) if cluster_id is not None else None

    # duration if present
    mdur = re.search(r"duration[_-](\d+)[_-]?days?", base)
    duration_days = int(mdur.group(1)) if mdur else None

    return {
        "event_file": base,
        "event_path": fp,
        "year": infer_year_from_path(fp),
        "start_date": start_date,
        "end_date": end_date,
        "duration_days": duration_days,
        "cluster_id": cluster_id,
    }

def commune_is_affected(src: rasterio.io.DatasetReader, geom, threshold_cm: float):
    """
    Returns (is_affected: bool, max_cm: float|None)
    """
    data, _ = mask(
        src, [geom],
        crop=True,
        filled=False,
        all_touched=ALL_TOUCHED,
        indexes=1
    )
    arr = data[0].astype(np.float32, copy=False)

    vals = arr.compressed() if np.ma.isMaskedArray(arr) else arr.ravel()
    if vals.size == 0:
        return False, None

    if IGNORE_VALUES:
        for v in IGNORE_VALUES:
            vals = vals[vals != v]
        if vals.size == 0:
            return False, None

    max_cm = float(np.max(vals))
    return (max_cm > threshold_cm), max_cm

# ============================================================
# Load communes + spatial index (WGS84 for bbox filtering)
# ============================================================

if COMMUNES_LAYER:
    communes = gpd.read_file(COMMUNES_PATH, layer=COMMUNES_LAYER)
else:
    communes = gpd.read_file(COMMUNES_PATH)

communes = communes[~communes.geometry.is_empty & communes.geometry.notna()].copy()
communes_wgs84 = communes.to_crs("EPSG:4326")
sindex = communes_wgs84.sindex

# ============================================================
# Collect all tifs from multiple roots
# ============================================================

tifs = []
for root in FLOODS_ROOTS:
    tifs.extend(glob.glob(os.path.join(root, "**", "*.tif"), recursive=True))
    tifs.extend(glob.glob(os.path.join(root, "**", "*.TIF"), recursive=True))

tifs = sorted(set(tifs))
print("Total TIFs found:", len(tifs))

# ============================================================
# Main loop
# ============================================================

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

wide_rows = []
long_rows = []

# diagnostics
diag = Counter()
error_counts = Counter()
ERROR_SAMPLE_LIMIT = 50
error_samples = []

for fp in tqdm(tifs, desc="Processing events"):
    meta = parse_event_from_name(fp)
    diag["events_total"] += 1
    if meta is None:
        diag["events_unparsed"] += 1
        continue
    diag["events_parsed"] += 1

    try:
        with rasterio.open(fp) as src:
            if src.crs is None:
                diag["events_no_crs"] += 1
                continue

            # raster bounds in WGS84 for candidate filtering
            b = transform_bounds(src.crs, "EPSG:4326", *src.bounds, densify_pts=21)
            raster_bbox_wgs84 = box(*b)

            # candidate communes by bbox using spatial index
            cand_idx = list(sindex.intersection(raster_bbox_wgs84.bounds))
            if not cand_idx:
                wide_rows.append({**meta, "n_communes": 0, "communes": ""})
                continue

            cands = communes_wgs84.iloc[cand_idx]
            cands = cands[cands.intersects(raster_bbox_wgs84)]
            if cands.empty:
                wide_rows.append({**meta, "n_communes": 0, "communes": ""})
                continue

            diag["events_with_candidates"] += 1

            # ---- CRITICAL FIX: reproject candidates to raster CRS BEFORE mask() ----
            cands_src = cands.to_crs(src.crs)

            # Optional but helpful: fix invalid geometries (Shapely 2+)
            # If your shapely/geopandas version doesn't support make_valid, comment this out.
            try:
                cands_src["geometry"] = cands_src.geometry.make_valid()
            except Exception:
                pass

            # cheap overlap check in raster CRS
            raster_box_src = box(*src.bounds)

            affected_communes = []
            any_affected = False

            for _, crow in cands_src.iterrows():
                geom = crow.geometry
                if geom is None or geom.is_empty:
                    continue

                # Skip truly non-overlapping shapes (prevents ValueError explosion)
                if not geom.intersects(raster_box_src):
                    continue

                try:
                    ok, max_cm = commune_is_affected(src, geom, threshold_cm=THRESHOLD_CM)
                except ValueError as e:
                    # most common: shapes do not overlap raster (should be rare now)
                    msg = f"commune_error: {repr(e)}"
                    error_counts[msg] += 1
                    if len(error_samples) < ERROR_SAMPLE_LIMIT:
                        error_samples.append((fp, msg))
                    continue
                except Exception as e:
                    msg = f"commune_error: {repr(e)}"
                    error_counts[msg] += 1
                    if len(error_samples) < ERROR_SAMPLE_LIMIT:
                        error_samples.append((fp, msg))
                    continue

                # collect long rows only when affected (you can change if you want all max_cm)
                if ok:
                    any_affected = True

                    cname = crow.get(COMMUNE_NAME_COL, None)
                    ccode = crow.get(COMMUNE_CODE_COL, None)

                    long_rows.append({
                        **meta,
                        "commune": cname,
                        "commune_code": ccode,
                        "max_cm": max_cm,
                    })

                    if cname is not None:
                        affected_communes.append(str(cname))

            # wide output: summary for the event
            wide_rows.append({
                **meta,
                "n_communes": len(affected_communes),
                "communes": "; ".join(affected_communes),
            })

            if any_affected:
                diag["events_with_affected"] += 1

    except Exception as e:
        msg = f"event_error: {repr(e)}"
        error_counts[msg] += 1
        if len(error_samples) < ERROR_SAMPLE_LIMIT:
            error_samples.append((fp, msg))
        continue

# ============================================================
# Build DataFrames + save
# ============================================================

wide_cols = [
    "year", "start_date", "end_date", "duration_days", "cluster_id",
    "event_file", "event_path",
    "n_communes", "communes"
]
long_cols = [
    "year", "start_date", "end_date", "duration_days", "cluster_id",
    "event_file", "event_path",
    "commune_code", "commune", "max_cm"
]

df_wide = pd.DataFrame(wide_rows, columns=wide_cols)
df_long = pd.DataFrame(long_rows, columns=long_cols)

if not df_wide.empty:
    df_wide = df_wide.sort_values(["year", "start_date", "end_date", "cluster_id"], na_position="last")

if not df_long.empty:
    df_long = df_long.sort_values(["year", "start_date", "end_date", "cluster_id", "commune"], na_position="last")

df_wide.to_csv(OUT_WIDE, index=False, encoding="utf-8-sig")
df_long.to_csv(OUT_LONG, index=False, encoding="utf-8-sig")

print("\nSaved combined outputs:")
print(" -", OUT_WIDE, "rows:", len(df_wide))
print(" -", OUT_LONG, "rows:", len(df_long))

print("\nDiagnostics:")
for k in ["events_total", "events_parsed", "events_with_candidates", "events_with_affected", "events_unparsed", "events_no_crs"]:
    if k in diag:
        print(" -", k + ":", diag[k])

print("\nError summary (top 10):")
for msg, n in error_counts.most_common(10):
    print(f" - {n}  {msg}")

if error_samples:
    print("\nError samples:")
    for fp, msg in error_samples[:10]:
        print(" -", os.path.basename(fp), "->", msg)


In [30]:
# ============================================================
# 3_Tabularization — FULL PIPELINE (AUTO-DETECT *_filtered)
# Builds:
#   - events_communes_wide_all_years.csv
#   - events_communes_long_all_years.csv
# ============================================================

import os, re, glob
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_bounds
from shapely.geometry import box
from tqdm import tqdm

# ============================================================
# CONFIG (EDIT ONLY THIS PART)
# ============================================================

FLOODS_BASE = r"data/JRC_flood_depth_maps"   # ROOT folder

COMMUNES_PATH = r"D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025.gpkg"
COMMUNES_LAYER = None        # or "communes"
COMMUNE_NAME_COL = "NOM"     # change if needed
COMMUNE_CODE_COL = "INSEE_COM"

OUT_DIR = r"data/JRC_flood_depth_maps/_outputs"
OUT_WIDE = os.path.join(OUT_DIR, "events_communes_wide_all_years.csv")
OUT_LONG = os.path.join(OUT_DIR, "events_communes_long_all_years.csv")

THRESHOLD_CM = 5.0           # flood threshold in cm
ALL_TOUCHED = True
IGNORE_VALUES = set()        # e.g. {0, -9999}

# ============================================================
# AUTO-DETECT *_filtered FOLDERS
# ============================================================

def find_flood_roots(base_dir: str | Path, suffix: str = "_filtered") -> list[str]:
    base_dir = Path(base_dir)
    roots = sorted(str(p) for p in base_dir.iterdir()
                   if p.is_dir() and p.name.endswith(suffix))

    if not roots:
        raise FileNotFoundError(f"No '*{suffix}' folders found in: {base_dir}")

    print("Detected FLOODS_ROOTS:")
    for r in roots:
        print(" -", r)
    return roots

FLOODS_ROOTS = find_flood_roots(FLOODS_BASE, suffix="_filtered")

# ============================================================
# HELPERS
# ============================================================

def infer_year_from_path(fp: str) -> int | None:
    m = re.search(r"(19|20)\d{2}", fp)
    return int(m.group(0)) if m else None

def parse_event_from_name(fp: str) -> dict | None:
    base = os.path.basename(fp)

    m = re.search(
        r"(?P<start>\d{4}-\d{2}-\d{2})---(?P<end>\d{4}-\d{2}-\d{2}).*?"
        r"(?:cluster[_-](?P<cluster>\d+))?",
        base
    )
    if not m:
        return None

    mdur = re.search(r"duration[_-](\d+)[_-]?days?", base)

    return {
        "event_file": base,
        "event_path": fp,
        "year": infer_year_from_path(fp),
        "start_date": m.group("start"),
        "end_date": m.group("end"),
        "duration_days": int(mdur.group(1)) if mdur else None,
        "cluster_id": int(m.group("cluster")) if m.group("cluster") else None,
    }

def commune_is_affected(src, geom, threshold_cm: float):
    data, _ = mask(
        src, [geom],
        crop=True,
        filled=False,
        all_touched=ALL_TOUCHED,
        indexes=1
    )

    arr = data[0].astype(np.float32, copy=False)
    vals = arr.compressed() if np.ma.isMaskedArray(arr) else arr.ravel()

    if vals.size == 0:
        return False, None

    if IGNORE_VALUES:
        for v in IGNORE_VALUES:
            vals = vals[vals != v]
        if vals.size == 0:
            return False, None

    max_cm = float(np.max(vals))
    return (max_cm > threshold_cm), max_cm

# ============================================================
# LOAD COMMUNES + SPATIAL INDEX (WGS84)
# ============================================================

communes = gpd.read_file(COMMUNES_PATH, layer=COMMUNES_LAYER)
communes = communes[~communes.geometry.is_empty & communes.geometry.notna()].copy()
communes_wgs84 = communes.to_crs("EPSG:4326")
sindex = communes_wgs84.sindex

# ============================================================
# COLLECT ALL TIFS
# ============================================================

tifs = []
for root in FLOODS_ROOTS:
    tifs.extend(glob.glob(os.path.join(root, "**", "*.tif"), recursive=True))
    tifs.extend(glob.glob(os.path.join(root, "**", "*.TIF"), recursive=True))

tifs = sorted(set(tifs))
print("\nTotal TIFs found:", len(tifs))

# ============================================================
# MAIN LOOP
# ============================================================

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

wide_rows = []
long_rows = []

diag = Counter()
error_counts = Counter()
ERROR_SAMPLE_LIMIT = 50
error_samples = []

for fp in tqdm(tifs, desc="Processing events"):
    meta = parse_event_from_name(fp)
    diag["events_total"] += 1

    if meta is None:
        diag["events_unparsed"] += 1
        continue
    diag["events_parsed"] += 1

    try:
        with rasterio.open(fp) as src:
            if src.crs is None:
                diag["events_no_crs"] += 1
                continue

            # --- STEP 1: fast filter in WGS84 ---
            b = transform_bounds(src.crs, "EPSG:4326", *src.bounds, densify_pts=21)
            raster_bbox_wgs84 = box(*b)

            cand_idx = list(sindex.intersection(raster_bbox_wgs84.bounds))
            if not cand_idx:
                wide_rows.append({**meta, "n_communes": 0, "communes": ""})
                continue

            cands = communes_wgs84.iloc[cand_idx]
            cands = cands[cands.intersects(raster_bbox_wgs84)]
            if cands.empty:
                wide_rows.append({**meta, "n_communes": 0, "communes": ""})
                continue

            diag["events_with_candidates"] += 1

            # --- STEP 2: reproject to raster CRS for MASKING ---
            cands_src = cands.to_crs(src.crs)

            try:
                cands_src["geometry"] = cands_src.geometry.make_valid()
            except Exception:
                pass

            raster_box_src = box(*src.bounds)

            affected_communes = []
            any_affected = False

            for _, crow in cands_src.iterrows():
                geom = crow.geometry
                if geom is None or geom.is_empty:
                    continue

                if not geom.intersects(raster_box_src):
                    continue

                try:
                    ok, max_cm = commune_is_affected(src, geom, THRESHOLD_CM)
                except Exception as e:
                    msg = f"commune_error: {repr(e)}"
                    error_counts[msg] += 1
                    if len(error_samples) < ERROR_SAMPLE_LIMIT:
                        error_samples.append((fp, msg))
                    continue

                if ok:
                    any_affected = True
                    cname = crow.get(COMMUNE_NAME_COL, None)
                    ccode = crow.get(COMMUNE_CODE_COL, None)

                    long_rows.append({
                        **meta,
                        "commune": cname,
                        "commune_code": ccode,
                        "max_cm": max_cm,
                    })

                    if cname:
                        affected_communes.append(str(cname))

            wide_rows.append({
                **meta,
                "n_communes": len(affected_communes),
                "communes": "; ".join(affected_communes),
            })

            if any_affected:
                diag["events_with_affected"] += 1

    except Exception as e:
        msg = f"event_error: {repr(e)}"
        error_counts[msg] += 1
        if len(error_samples) < ERROR_SAMPLE_LIMIT:
            error_samples.append((fp, msg))
        continue

# ============================================================
# BUILD DATAFRAMES + SAVE
# ============================================================

wide_cols = [
    "year","start_date","end_date","duration_days","cluster_id",
    "event_file","event_path","n_communes","communes"
]

long_cols = [
    "year","start_date","end_date","duration_days","cluster_id",
    "event_file","event_path","commune_code","commune","max_cm"
]

df_wide = pd.DataFrame(wide_rows, columns=wide_cols)
df_long = pd.DataFrame(long_rows, columns=long_cols)

if not df_wide.empty:
    df_wide = df_wide.sort_values(["year","start_date","end_date","cluster_id"])

if not df_long.empty:
    df_long = df_long.sort_values(["year","start_date","end_date","cluster_id","commune"])

df_wide.to_csv(OUT_WIDE, index=False, encoding="utf-8-sig")
df_long.to_csv(OUT_LONG, index=False, encoding="utf-8-sig")

print("\nSaved:")
print(" -", OUT_WIDE, "rows:", len(df_wide))
print(" -", OUT_LONG, "rows:", len(df_long))

print("\nDiagnostics:")
for k in ["events_total","events_parsed","events_with_candidates","events_with_affected"]:
    print(f" - {k}: {diag[k]}")

print("\nTop error types:")
for msg, n in error_counts.most_common(5):
    print(f" - {n}  {msg}")


Detected FLOODS_ROOTS:
 - data\JRC_flood_depth_maps\2015_filtered
 - data\JRC_flood_depth_maps\2016_filtered
 - data\JRC_flood_depth_maps\2017_filtered
 - data\JRC_flood_depth_maps\2018_filtered
 - data\JRC_flood_depth_maps\2019_filtered
 - data\JRC_flood_depth_maps\2020_filtered
 - data\JRC_flood_depth_maps\2021_filtered
 - data\JRC_flood_depth_maps\2022_filtered
 - data\JRC_flood_depth_maps\2023_filtered
 - data\JRC_flood_depth_maps\2024_filtered


D:\M2_MoSEF\Acclim\DataCollection\.venv\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'adminexpress-cog-simpl-000-2025.gpkg': 'commune_int' (default), 'commune', 'departement', 'region', 'cheflieu', 'epci', 'departement_int', 'region_int', 'epci_int'. Specify layer parameter to avoid this warning.
  result = read_func(



Total TIFs found: 1880


Processing events:   0%|▋                                                                                                                                                                                    | 7/1880 [01:03<4:44:22,  9.11s/it]


KeyboardInterrupt: 

In [31]:
# ============================================================
# FULL FUNCTIONAL PIPELINE — European Satellite-Derived Flood Depth Maps
# (README-confirmed: depths in cm, uint16, 9999 dummy permanent/seasonal water)
#
# - Auto-detects all *_filtered folders under FLOODS_BASE
# - Reads AdminExpress communes from your GPKG + layer
# - Filters candidates in EPSG:4326 (fast sindex), masks in src.crs (correct)
# - Creates combined outputs across all years:
#     events_communes_wide_all_years.csv
#     events_communes_long_all_years.csv
# ============================================================

import os, re, glob
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_bounds
from shapely.geometry import box
from tqdm import tqdm

# =========================
# CONFIG (your paths)
# =========================
FLOODS_BASE = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps"

COMMUNES_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025.gpkg"
COMMUNES_LAYER = "commune"  # polygon layer

# AdminExpress columns (adjust if your layer uses uppercase)
COMMUNE_NAME_COL = "nom"
COMMUNE_CODE_COL = "insee_com"

OUT_DIR = os.path.join(FLOODS_BASE, "_outputs")
OUT_WIDE = os.path.join(OUT_DIR, "events_communes_wide_all_years.csv")
OUT_LONG = os.path.join(OUT_DIR, "events_communes_long_all_years.csv")

# README: depths in cm
THRESHOLD_CM = 0.0          # affected if max depth > 0 cm
IGNORE_VALUES = {9999}      # dummy value = permanent/seasonal water bodies
ALL_TOUCHED = True

# reduce memory blow-ups: count errors, store only a sample
ERROR_SAMPLE_LIMIT = 50

# =========================
# Helpers
# =========================

def find_flood_roots(base_dir: str | Path, suffix: str = "_filtered") -> list[str]:
    base_dir = Path(base_dir)
    if not base_dir.exists():
        raise FileNotFoundError(f"FLOODS_BASE not found: {base_dir}")
    roots = sorted(str(p) for p in base_dir.iterdir() if p.is_dir() and p.name.endswith(suffix))
    if not roots:
        raise FileNotFoundError(f"No '*{suffix}' folders found under: {base_dir}")
    return roots

def collect_tifs(roots: list[str]) -> list[str]:
    tifs = []
    for root in roots:
        tifs.extend(glob.glob(os.path.join(root, "**", "*.tif"), recursive=True))
        tifs.extend(glob.glob(os.path.join(root, "**", "*.TIF"), recursive=True))
    return sorted(set(tifs))

def infer_year_from_path(fp: str) -> int | None:
    m = re.search(r"(19|20)\d{2}", fp)
    return int(m.group(0)) if m else None

def parse_event_from_name(fp: str) -> dict | None:
    """
    README naming:
      WD_MERGE_[start]---[end]_duration_[days]_days_cluster_[id]_...
    We'll parse start/end/cluster and duration if present.
    """
    base = os.path.basename(fp)
    m = re.search(
        r"(?P<start>\d{4}-\d{2}-\d{2})---(?P<end>\d{4}-\d{2}-\d{2}).*?"
        r"(?:cluster[_-](?P<cluster>\d+))?",
        base
    )
    if not m:
        return None

    mdur = re.search(r"duration[_-](\d+)[_-]?days?", base)
    return {
        "event_file": base,
        "event_path": fp,
        "year": infer_year_from_path(fp),
        "start_date": m.group("start"),
        "end_date": m.group("end"),
        "duration_days": int(mdur.group(1)) if mdur else None,
        "cluster_id": int(m.group("cluster")) if m.group("cluster") else None,
    }

def commune_is_affected(src, geom, threshold_cm: float):
    """
    Returns (affected: bool, max_cm: float|None)
    NOTE: geom MUST be in src.crs
    """
    data, _ = mask(
        src, [geom],
        crop=True,
        filled=False,
        all_touched=ALL_TOUCHED,
        indexes=1
    )
    arr = data[0].astype(np.float32, copy=False)
    vals = arr.compressed() if np.ma.isMaskedArray(arr) else arr.ravel()
    if vals.size == 0:
        return False, None

    # remove dummy values (README: 9999 = water bodies)
    if IGNORE_VALUES:
        for v in IGNORE_VALUES:
            vals = vals[vals != v]
        if vals.size == 0:
            return False, None

    max_cm = float(np.max(vals))
    return (max_cm > threshold_cm), max_cm

# =========================
# Load communes (AdminExpress)
# =========================
communes = gpd.read_file(COMMUNES_GPKG, layer=COMMUNES_LAYER)
communes = communes[communes.geometry.notna() & ~communes.geometry.is_empty].copy()

# We do candidate filtering in EPSG:4326 (fast)
communes_wgs84 = communes.to_crs("EPSG:4326")
sindex = communes_wgs84.sindex

# =========================
# Discover floods + collect tifs
# =========================
roots = find_flood_roots(FLOODS_BASE, suffix="_filtered")
print("Detected flood folders:")
for r in roots:
    print(" -", r)

tifs = collect_tifs(roots)
print("\nTotal TIFs found:", len(tifs))

# =========================
# Main loop
# =========================
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

wide_rows = []
long_rows = []

diag = Counter()
error_counts = Counter()
error_samples = []

for fp in tqdm(tifs, desc="Processing events"):
    diag["events_total"] += 1
    meta = parse_event_from_name(fp)
    if meta is None:
        diag["events_unparsed"] += 1
        continue
    diag["events_parsed"] += 1

    try:
        with rasterio.open(fp) as src:
            if src.crs is None:
                diag["events_no_crs"] += 1
                continue

            # ---- STEP 1: bbox filtering in WGS84 (EPSG:4326) ----
            b = transform_bounds(src.crs, "EPSG:4326", *src.bounds, densify_pts=21)
            raster_bbox_wgs84 = box(*b)

            cand_idx = list(sindex.intersection(raster_bbox_wgs84.bounds))
            if not cand_idx:
                wide_rows.append({**meta, "n_communes": 0, "communes": ""})
                continue

            cands = communes_wgs84.iloc[cand_idx]
            cands = cands[cands.intersects(raster_bbox_wgs84)]
            if cands.empty:
                wide_rows.append({**meta, "n_communes": 0, "communes": ""})
                continue

            # candidates exist (in 4326)
            diag["events_with_candidates_bbox"] += 1

            # ---- STEP 2: reproject candidates to raster CRS for mask() ----
            cands_src = cands.to_crs(src.crs)

            # make geometries valid if possible (Shapely 2)
            try:
                cands_src["geometry"] = cands_src.geometry.make_valid()
            except Exception:
                pass

            raster_box_src = box(*src.bounds)

            affected_names = []
            any_affected = False
            tested = 0
            overlap_skips = 0

            for _, crow in cands_src.iterrows():
                geom = crow.geometry
                if geom is None or geom.is_empty:
                    continue

                # cheap overlap guard to prevent ValueError explosion
                if not geom.intersects(raster_box_src):
                    overlap_skips += 1
                    continue

                tested += 1
                try:
                    ok, max_cm = commune_is_affected(src, geom, THRESHOLD_CM)
                except Exception as e:
                    msg = f"commune_error: {repr(e)}"
                    error_counts[msg] += 1
                    if len(error_samples) < ERROR_SAMPLE_LIMIT:
                        error_samples.append((fp, msg))
                    continue

                if ok:
                    any_affected = True
                    cname = crow.get(COMMUNE_NAME_COL, None)
                    ccode = crow.get(COMMUNE_CODE_COL, None)

                    long_rows.append({
                        **meta,
                        "commune": cname,
                        "commune_code": ccode,
                        "max_cm": max_cm,
                    })
                    if cname:
                        affected_names.append(str(cname))

            wide_rows.append({
                **meta,
                "n_communes": len(affected_names),
                "communes": "; ".join(affected_names),
                "n_candidates_tested": tested,
                "n_overlap_skipped": overlap_skips,
            })

            if any_affected:
                diag["events_with_affected"] += 1

    except Exception as e:
        msg = f"event_error: {repr(e)}"
        error_counts[msg] += 1
        if len(error_samples) < ERROR_SAMPLE_LIMIT:
            error_samples.append((fp, msg))
        continue

# =========================
# Build DataFrames + save
# =========================
wide_cols = [
    "year","start_date","end_date","duration_days","cluster_id",
    "event_file","event_path",
    "n_communes","communes",
    "n_candidates_tested","n_overlap_skipped"
]
long_cols = [
    "year","start_date","end_date","duration_days","cluster_id",
    "event_file","event_path",
    "commune_code","commune","max_cm"
]

df_wide = pd.DataFrame(wide_rows, columns=wide_cols)
df_long = pd.DataFrame(long_rows, columns=long_cols)

if not df_wide.empty:
    df_wide = df_wide.sort_values(["year","start_date","end_date","cluster_id"], na_position="last")
if not df_long.empty:
    df_long = df_long.sort_values(["year","start_date","end_date","cluster_id","commune"], na_position="last")

df_wide.to_csv(OUT_WIDE, index=False, encoding="utf-8-sig")
df_long.to_csv(OUT_LONG, index=False, encoding="utf-8-sig")

print("\nSaved combined outputs:")
print(" -", OUT_WIDE, "rows:", len(df_wide))
print(" -", OUT_LONG, "rows:", len(df_long))

print("\nDiagnostics:")
for k in ["events_total","events_parsed","events_unparsed","events_no_crs","events_with_candidates_bbox","events_with_affected"]:
    if k in diag:
        print(" -", k + ":", diag[k])

print("\nTop error types:")
for msg, n in error_counts.most_common(10):
    print(f" - {n}  {msg}")

if error_samples:
    print("\nError samples:")
    for fp, msg in error_samples[:10]:
        print(" -", os.path.basename(fp), "->", msg)


Detected flood folders:
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2015_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2016_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2017_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2018_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2019_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2020_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2021_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2022_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2023_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered

Total TIFs found: 1880


Processing events:  12%|██████████████████████▎                                                                                                                                                            | 234/1880 [21:33<2:31:40,  5.53s/it]


KeyboardInterrupt: 

In [32]:
import os, re, glob
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_bounds
from shapely.geometry import box
from tqdm import tqdm

# =========================
# CONFIG (YOUR PATHS)
# =========================
FLOODS_BASE = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps"

COMMUNES_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025.gpkg"
COMMUNES_LAYER = "commune"

COMMUNE_NAME_COL = "nom"       # adjust if uppercase in your file
COMMUNE_CODE_COL = "insee_com" # adjust if uppercase in your file

OUT_DIR = os.path.join(FLOODS_BASE, "_outputs")
OUT_WIDE = os.path.join(OUT_DIR, "events_communes_wide_all_years.csv")
OUT_LONG = os.path.join(OUT_DIR, "events_communes_long_all_years.csv")

# README: depths are in cm, dtype uint16, dummy 9999 = water bodies
THRESHOLD_CM = 0.0
IGNORE_VALUES = {9999}
ALL_TOUCHED = True

ERROR_SAMPLE_LIMIT = 50

# =========================
# HELPERS
# =========================
def find_flood_roots(base_dir: str | Path, suffix: str = "_filtered") -> list[str]:
    base_dir = Path(base_dir)
    if not base_dir.exists():
        raise FileNotFoundError(f"FLOODS_BASE not found: {base_dir}")
    roots = sorted(str(p) for p in base_dir.iterdir() if p.is_dir() and p.name.endswith(suffix))
    if not roots:
        raise FileNotFoundError(f"No '*{suffix}' folders found under: {base_dir}")
    return roots

def collect_tifs(roots: list[str]) -> list[str]:
    tifs = []
    for root in roots:
        tifs.extend(glob.glob(os.path.join(root, "**", "*.tif"), recursive=True))
        tifs.extend(glob.glob(os.path.join(root, "**", "*.TIF"), recursive=True))
    return sorted(set(tifs))

def infer_year_from_path(fp: str) -> int | None:
    m = re.search(r"(19|20)\d{2}", fp)
    return int(m.group(0)) if m else None

def parse_event_from_name(fp: str) -> dict | None:
    """
    README naming:
      WD_MERGE_[start]---[end]_duration_[days]_days_cluster_[id]_...
    """
    base = os.path.basename(fp)
    m = re.search(
        r"(?P<start>\d{4}-\d{2}-\d{2})---(?P<end>\d{4}-\d{2}-\d{2}).*?"
        r"(?:cluster[_-](?P<cluster>\d+))?",
        base
    )
    if not m:
        return None
    mdur = re.search(r"duration[_-](\d+)[_-]?days?", base)

    start_date = m.group("start")
    end_date = m.group("end")
    cluster_id = int(m.group("cluster")) if m.group("cluster") else None

    return {
        "event_file": base,
        "event_path": fp,
        "year": infer_year_from_path(fp),
        "start_date": start_date,
        "end_date": end_date,
        "duration_days": int(mdur.group(1)) if mdur else None,
        "cluster_id": cluster_id,
    }

def event_id_from_meta(meta: dict) -> str:
    # stable id for grouping tiles belonging to same event
    cid = meta["cluster_id"]
    return f'{meta["start_date"]}---{meta["end_date"]}_cluster_{cid}'

def commune_is_affected(src, geom, threshold_cm: float):
    """
    Returns (affected: bool, max_cm: float|None)
    NOTE: geom MUST be in src.crs
    """
    data, _ = mask(
        src, [geom],
        crop=True,
        filled=False,
        all_touched=ALL_TOUCHED,
        indexes=1
    )
    arr = data[0].astype(np.float32, copy=False)
    vals = arr.compressed() if np.ma.isMaskedArray(arr) else arr.ravel()

    if vals.size == 0:
        return False, None

    if IGNORE_VALUES:
        for v in IGNORE_VALUES:
            vals = vals[vals != v]
        if vals.size == 0:
            return False, None

    max_cm = float(np.max(vals))
    return (max_cm > threshold_cm), max_cm

# =========================
# LOAD COMMUNES (AdminExpress)
# =========================
communes = gpd.read_file(COMMUNES_GPKG, layer=COMMUNES_LAYER)
communes = communes[communes.geometry.notna() & ~communes.geometry.is_empty].copy()

# Candidate filtering in EPSG:4326
communes_wgs84 = communes.to_crs("EPSG:4326")
sindex = communes_wgs84.sindex

# =========================
# DISCOVER FLOODS + GROUP FILES BY EVENT
# =========================
roots = find_flood_roots(FLOODS_BASE, suffix="_filtered")
print("Detected flood folders:")
for r in roots:
    print(" -", r)

tifs = collect_tifs(roots)
print("\nTotal TIFs found:", len(tifs))

event_to_files = defaultdict(list)
event_meta = {}  # event_id -> meta (representative)

unparsed = 0
for fp in tifs:
    meta = parse_event_from_name(fp)
    if meta is None:
        unparsed += 1
        continue
    eid = event_id_from_meta(meta)
    event_to_files[eid].append(fp)
    # store one meta per event
    if eid not in event_meta:
        meta2 = meta.copy()
        meta2["event_id"] = eid
        event_meta[eid] = meta2

print("Unparsed files:", unparsed)
print("Distinct events:", len(event_to_files))

# =========================
# MAIN LOOP (per EVENT, multiple tiles)
# =========================
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

wide_rows = []
long_rows = []

diag = Counter()
error_counts = Counter()
error_samples = []

for eid, files in tqdm(event_to_files.items(), desc="Processing events"):
    diag["events_total"] += 1
    meta = event_meta[eid]

    # Track max depth per commune across all tiles
    commune_max = {}  # key -> max_cm
    tested_total = 0
    overlap_skips_total = 0
    any_candidates = False

    # process each tile for the event
    for fp in files:
        try:
            with rasterio.open(fp) as src:
                if src.crs is None:
                    diag["tiles_no_crs"] += 1
                    continue

                # ---- STEP 1: bbox filtering in WGS84 ----
                b = transform_bounds(src.crs, "EPSG:4326", *src.bounds, densify_pts=21)
                raster_bbox_wgs84 = box(*b)

                cand_idx = list(sindex.intersection(raster_bbox_wgs84.bounds))
                if not cand_idx:
                    continue

                cands = communes_wgs84.iloc[cand_idx]
                cands = cands[cands.intersects(raster_bbox_wgs84)]
                if cands.empty:
                    continue

                any_candidates = True

                # ---- STEP 2: reproject candidates to raster CRS for mask() ----
                cands_src = cands.to_crs(src.crs)

                # make valid if available (Shapely 2)
                try:
                    cands_src["geometry"] = cands_src.geometry.make_valid()
                except Exception:
                    pass

                raster_box_src = box(*src.bounds)

                for _, crow in cands_src.iterrows():
                    geom = crow.geometry
                    if geom is None or geom.is_empty:
                        continue

                    if not geom.intersects(raster_box_src):
                        overlap_skips_total += 1
                        continue

                    tested_total += 1

                    try:
                        ok, max_cm = commune_is_affected(src, geom, THRESHOLD_CM)
                    except Exception as e:
                        msg = f"commune_error: {repr(e)}"
                        error_counts[msg] += 1
                        if len(error_samples) < ERROR_SAMPLE_LIMIT:
                            error_samples.append((fp, msg))
                        continue

                    if not ok:
                        continue

                    cname = crow.get(COMMUNE_NAME_COL, None)
                    ccode = crow.get(COMMUNE_CODE_COL, None)

                    key = (ccode if ccode is not None else cname)
                    if key is None:
                        # fallback key if both missing
                        key = f"row_{_}"

                    prev = commune_max.get(key)
                    if prev is None or max_cm > prev:
                        commune_max[key] = max_cm

        except Exception as e:
            msg = f"tile_error: {repr(e)}"
            error_counts[msg] += 1
            if len(error_samples) < ERROR_SAMPLE_LIMIT:
                error_samples.append((fp, msg))
            continue

    if any_candidates:
        diag["events_with_candidates_bbox"] += 1

    # build long rows for the event (max over tiles)
    affected_names = []
    for key, max_cm in commune_max.items():
        # recover name/code for output: re-lookup in communes table if possible
        # simplest: just output key as code if looks like INSEE code; else as name
        ccode_out = None
        cname_out = None

        if isinstance(key, str) and key.isdigit() and len(key) in (5,):
            ccode_out = key
        else:
            # could be actual code (string) or name; try both
            ccode_out = key if key is not None else None

        # best effort name: if we have code column, join back
        if COMMUNE_CODE_COL in communes.columns and ccode_out is not None:
            match = communes.loc[communes[COMMUNE_CODE_COL].astype(str) == str(ccode_out)]
            if not match.empty and COMMUNE_NAME_COL in match.columns:
                cname_out = match.iloc[0][COMMUNE_NAME_COL]
        if cname_out is None and isinstance(key, str):
            cname_out = key

        long_rows.append({
            **meta,
            "commune_code": ccode_out,
            "commune": cname_out,
            "max_cm": float(max_cm),
        })
        if cname_out:
            affected_names.append(str(cname_out))

    # one wide row per EVENT
    wide_rows.append({
        **meta,
        "n_communes": len(commune_max),
        "communes": "; ".join(sorted(set(affected_names))),
        "n_tiles": len(files),
        "n_candidates_tested": tested_total,
        "n_overlap_skipped": overlap_skips_total,
    })

    if commune_max:
        diag["events_with_affected"] += 1

# =========================
# BUILD DATAFRAMES + SAVE
# =========================
wide_cols = [
    "event_id","year","start_date","end_date","duration_days","cluster_id",
    "n_tiles","event_path",  # event_path here is representative (first tile)
    "n_communes","communes",
    "n_candidates_tested","n_overlap_skipped"
]
long_cols = [
    "event_id","year","start_date","end_date","duration_days","cluster_id",
    "commune_code","commune","max_cm"
]

df_wide = pd.DataFrame(wide_rows, columns=wide_cols)
df_long = pd.DataFrame(long_rows, columns=long_cols)

if not df_wide.empty:
    df_wide = df_wide.sort_values(["year","start_date","end_date","cluster_id"], na_position="last")
if not df_long.empty:
    df_long = df_long.sort_values(["year","start_date","end_date","cluster_id","commune"], na_position="last")

df_wide.to_csv(OUT_WIDE, index=False, encoding="utf-8-sig")
df_long.to_csv(OUT_LONG, index=False, encoding="utf-8-sig")

print("\nSaved combined outputs:")
print(" -", OUT_WIDE, "rows:", len(df_wide))
print(" -", OUT_LONG, "rows:", len(df_long))

print("\nDiagnostics:")
for k in ["events_total","events_with_candidates_bbox","events_with_affected","tiles_no_crs"]:
    if k in diag:
        print(" -", k + ":", diag[k])

print("\nTop error types:")
for msg, n in error_counts.most_common(10):
    print(f" - {n}  {msg}")

if error_samples:
    print("\nError samples:")
    for fp, msg in error_samples[:10]:
        print(" -", os.path.basename(fp), "->", msg)


Detected flood folders:
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2015_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2016_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2017_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2018_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2019_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2020_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2021_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2022_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2023_filtered
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered

Total TIFs found: 1880
Unparsed files: 0
Distinct events: 539


Processing events: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 539/539 [3:02:56<00:00, 20.36s/it]


Saved combined outputs:
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs\events_communes_wide_all_years.csv rows: 539
 - D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\_outputs\events_communes_long_all_years.csv rows: 5378

Diagnostics:
 - events_total: 539
 - events_with_candidates_bbox: 413
 - events_with_affected: 129

Top error types:


In [35]:
error_samples


[]